# Pipeline de ingestión SQL con validación de datos Automatizado

In [ ]:
import pandas as pd
import sqlite3
import hashlib
import logging
import os
import shutil
import sys
from datetime import datetime

# ==========================================
# CONFIGURACIÓN DE ENTORNO Y LOGGING
# ==========================================
PATH_LANDING = "./data/landing/"
PATH_ARCHIVE = "./data/archive/"
DB_NAME = "../db/prueba_sql.db"

# Solución para NameError: __file__ (Compatible con Notebooks y Scripts)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

LOG_FILE = os.path.join(BASE_DIR, 'etl_pipeline_pro.log')

# Configuración Maestra de Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ],
    force=True 
)

In [2]:
class DataEngineerPipeline:
    def __init__(self, db_path):
        self.db_path = db_path
        self._prepare_environment()

    def _prepare_environment(self):
        """Asegura que existan las carpetas de trabajo"""
        for path in [PATH_LANDING, PATH_ARCHIVE]:
            if not os.path.exists(path):
                os.makedirs(path)
        logging.info("--- SISTEMA INICIALIZADO: Carpetas verificadas ---")

    def generate_hash_id(self, df, columns_to_hash):
        """Crea una firma digital (SHA-256) para cada fila para evitar duplicados"""
        # fillna('NULL') garantiza que el hash sea consistente si hay vacíos
        combined = df[columns_to_hash].fillna('NULL').astype(str).agg('-'.join, axis=1)
        df['idingesta'] = [hashlib.sha256(val.encode()).hexdigest() for val in combined]
        return df

    def anonymize_data(self, df, table_key):
        """Protección de datos sensibles (PII) usando métodos vectorizados"""
        if table_key == "almacenes" and 'almacencodigo' in df.columns:
            df['nombre'] = "ALMACEN_REF_" + df['almacencodigo'].astype(str)
        
        elif table_key == "personas":
            for col in ['primernombre', 'primerapellido']:
                if col in df.columns:
                    # Acceso directo a strings (mucho más rápido que .apply)
                    df[col] = df[col].astype(str).str[0].fillna('') + "***"
            if 'celular' in df.columns:
                df['celular'] = "****" + df['celular'].astype(str).str[-4:]
        return df

    def bulk_upsert(self, df, table_name):
        """Carga masiva deduplicada mediante tabla de Staging"""
        conn = sqlite3.connect(self.db_path)
        staging_table = f"stg_{table_name}"
        
        try:
            # Carga rápida por bloques
            df.to_sql(staging_table, conn, if_exists='replace', index=False, chunksize=10000)

            # Asegurar que la tabla destino exista con el mismo esquema
            conn.execute(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM {staging_table} WHERE 1=0")
            conn.execute(f"CREATE UNIQUE INDEX IF NOT EXISTS idx_{table_name}_hash ON {table_name}(idingesta)")

            # Inserción Incremental: Solo lo que no existe en la tabla final
            query = f"""
                INSERT INTO {table_name}
                SELECT * FROM {staging_table}
                WHERE idingesta NOT IN (SELECT idingesta FROM {table_name})
            """
            cursor = conn.execute(query)
            rows_added = cursor.rowcount
            
            conn.execute(f"DROP TABLE {staging_table}")
            conn.commit()
            return rows_added
        except Exception as e:
            logging.error(f"Error en Upsert para {table_name}: {e}")
            return 0
        finally:
            conn.close()

    def run(self):
        """Orquestador del Pipeline con Schema-on-Read y Robustez de Delimitadores"""
        ESQUEMAS = {
            "departamentos": ["Codigo", "PaisCodigo", "Nombre"],
            "municipios": ["MunicipioCodigo", "PaisCodigo", "DepartamentoCodigo", "Nombre"],
            "personas": [
                "personacodigo", "primernombre", "segundonombre", "primerapellido", "segundoapellido",
                "cupototal", "cupodisponibletotal", "departamentocodigo", "municipiocodigo",
                "fechaexpedicion", "fechanacimiento", "genero", "celular", "personasacargo"
            ],
            "almacenes": ["almacencodigo", "nombre"],
            "creditos": [
                "personacodigo", "almacencodigo", "valorfactura", "fechacreacion",
                "saldocapital", "saldofinanciacion"
            ]
        }

        archivos = [f for f in os.listdir(PATH_LANDING) if f.lower().endswith('.csv')]
        
        if not archivos:
            logging.info("Finalizado: No hay archivos .csv nuevos en /landing.")
            return

        for archivo in archivos:
            start_time = datetime.now()
            ruta_csv = os.path.join(PATH_LANDING, archivo)
            
            # Identificar el esquema correcto basado en el nombre del archivo
            tipo_tabla = next((k for k in ESQUEMAS.keys() if k in archivo.lower()), None)
            
            if not tipo_tabla:
                logging.warning(f"Omitiendo archivo desconocido: {archivo}")
                continue

            try:
                logging.info(f"Procesando archivo: {archivo}")
                
                # LECTURA ROBUSTA: Soporta mezcla de ',' y ';' automáticamente
                df = pd.read_csv(
                    ruta_csv,
                    header=None,
                    names=ESQUEMAS[tipo_tabla],
                    sep=r'[;,]', 
                    engine='python', 
                    on_bad_lines='warn',
                    na_values=['NULL', 'nan', '', 'None'],
                    encoding='latin-1'
                )

                # Selección de columnas para la llave única (deduplicación)
                if tipo_tabla == "creditos":
                    cols_hash = ['personacodigo', 'almacencodigo', 'valorfactura', 'fechacreacion']
                elif tipo_tabla in ["personas", "almacenes", "municipios", "departamentos"]:
                    cols_hash = [ESQUEMAS[tipo_tabla][0]] 
                else:
                    cols_hash = ESQUEMAS[tipo_tabla]

                # --- TRANSFORMACIONES ---
                df = self.generate_hash_id(df, cols_hash)
                df = self.anonymize_data(df, tipo_tabla)
                
                # --- CARGA ---
                registros = self.bulk_upsert(df, tipo_tabla)
                
                # --- GESTIÓN DE ARCHIVOS ---
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                shutil.move(ruta_csv, os.path.join(PATH_ARCHIVE, f"{timestamp}_{archivo}"))
                
                duracion = (datetime.now() - start_time).total_seconds()
                logging.info(f"OK: {archivo} | {registros} registros nuevos | {duracion:.2f} segundos.")

                del df # Limpieza de memoria

            except Exception as e:
                logging.error(f"Error procesando {archivo}: {str(e)}")


#### Antes de hacer carga de datos a la base de datos, se realiza pruebas:
    - Pruebas de integridad del pipeline, para asegurar el tratamiento de datos.
    - Pruebas de calidad de datos crudos, que la estructura sea compatible con las tablas de la base de datos.

### Prueba de pipeline (que no tenga cambios o errores generados en la manipulación de código)



In [3]:
import unittest
import pandas as pd
import io

class TestETLPipeline(unittest.TestCase):
    
    def setUp(self):
        """Configuración de datos mock para las pruebas"""
        # No necesitamos una DB real para probar la lógica de transformación
        self.pipeline = DataEngineerPipeline("test_temp.db")
        
        self.mock_data = pd.DataFrame({
            'personacodigo': [101, 102],
            'primernombre': ['JUAN', 'CARLOS'],
            'primerapellido': ['TORO', 'BETANCUR'],
            'celular': ['3001234567', '3159876543']
        })

    def test_anonymization_logic(self):
        """Verifica que la PII (Nombres y Celulares) se enmascare correctamente"""
        df_result = self.pipeline.anonymize_data(self.mock_data.copy(), "personas")
        
        # Validar nombres: Primera letra + ***
        self.assertEqual(df_result.iloc[0]['primernombre'], "J***")
        self.assertEqual(df_result.iloc[1]['primernombre'], "C***")
        
        # Validar celular: **** + últimos 4 dígitos
        self.assertEqual(df_result.iloc[0]['celular'], "****4567")
        self.assertEqual(df_result.iloc[1]['celular'], "****6543")

    def test_hash_generation(self):
        """Verifica que la firma digital 'idingesta' sea un SHA-256 válido"""
        df_result = self.pipeline.generate_hash_id(self.mock_data.copy(), ['personacodigo'])
        
        # Verificar que la columna existe
        self.assertIn('idingesta', df_result.columns)
        
        # Verificar longitud del hash SHA-256 (64 caracteres hexadecimales)
        self.assertEqual(len(df_result.iloc[0]['idingesta']), 64)
        
        # Verificar que sean únicos para datos diferentes
        self.assertNotEqual(df_result.iloc[0]['idingesta'], df_result.iloc[1]['idingesta'])

    def test_schema_robustness(self):
        """Prueba que el proceso no falle si hay valores nulos en columnas de hash"""
        df_nulls = pd.DataFrame({'personacodigo': [None], 'nombre': ['TEST']})
        # No debe lanzar excepción al hashear nulos
        try:
            self.pipeline.generate_hash_id(df_nulls, ['personacodigo'])
            success = True
        except Exception:
            success = False
        self.assertTrue(success)

# --- FUNCIÓN PARA EJECUTAR LOS TESTS DENTRO DEL NOTEBOOK ---
def run_pipeline_tests():
    print("🔍 Iniciando validación de calidad de código...")
    suite = unittest.TestLoader().loadTestsFromTestCase(TestETLPipeline)
    # Usamos un stream para que la salida se vea limpia en la celda
    runner = unittest.TextTestRunner(verbosity=1, stream=sys.stdout)
    result = runner.run(suite)
    
    if result.wasSuccessful():
        print("\n✅ TODAS LAS PRUEBAS PASARON: La lógica de transformación es segura.")
    else:
        print("\n❌ ALGUNAS PRUEBAS FALLARON: Revisa la lógica de anonimización o hashing.")

# Ejecutar las pruebas
run_pipeline_tests()

🔍 Iniciando validación de calidad de código...
2026-03-29 11:20:16,078 - INFO - --- SISTEMA INICIALIZADO: Carpetas verificadas ---
.2026-03-29 11:20:16,080 - INFO - --- SISTEMA INICIALIZADO: Carpetas verificadas ---
.2026-03-29 11:20:16,088 - INFO - --- SISTEMA INICIALIZADO: Carpetas verificadas ---
.
----------------------------------------------------------------------
Ran 3 tests in 0.015s

OK

✅ TODAS LAS PRUEBAS PASARON: La lógica de transformación es segura.


### Prueba de calidad de datos

In [4]:
# ==========================================
# EXTENSIÓN DE CALIDAD DE DATOS (DATA QUALITY)
# ==========================================

def validate_data_quality(self, df, table_name, expected_count):
    """
    Función que se inyectará en la clase existente.
    Verifica que el archivo real cumpla con el esquema.
    """
    actual_count = len(df.columns)
    if actual_count != expected_count:
        error_msg = f"❌ FALLO DE CALIDAD en {table_name}: Se esperaban {expected_count} columnas, pero llegaron {actual_count}."
        logging.error(error_msg)
        raise ValueError(error_msg)
    
    if df.empty:
        error_msg = f"❌ FALLO DE CALIDAD en {table_name}: El archivo está totalmente vacío."
        logging.error(error_msg)
        raise ValueError(error_msg)
        
    logging.info(f"✅ Calidad OK para {table_name} ({actual_count} columnas detectadas).")
    return True

# Inyectamos el método dinámicamente en la clase original
DataEngineerPipeline.validate_data_quality = validate_data_quality

print("🛠️ Método 'validate_data_quality' inyectado exitosamente en el Pipeline.")

🛠️ Método 'validate_data_quality' inyectado exitosamente en el Pipeline.


In [5]:
def task_data_quality_control():
    """
    Valida la estructura física de los CSVs en /landing.
    Nota: No incluye 'idingesta' porque esa columna se genera después.
    """
    pipeline_val = DataEngineerPipeline(DB_NAME)
    
    # METADATA: Columnas reales que DEBEN venir en el archivo físico
    # (Restamos 1 a las tablas que llevan Hash si antes contabas el hash)
    METADATA_FISICA = {
        "departamentos": 3,
        "municipios": 4,
        "personas": 14, # 14 en el CSV + 1 hash que creamos luego = 15 total
        "almacenes": 2,
        "creditos": 6   # 6 en el CSV + 1 hash que creamos luego = 7 total
    }
    
    archivos = [f for f in os.listdir(PATH_LANDING) if f.lower().endswith('.csv')]
    
    print(f"🔍 Auditando {len(archivos)} archivos físicos en landing...")
    
    for archivo in archivos:
        tipo_tabla = next((k for k in METADATA_FISICA.keys() if k in archivo.lower()), None)
        if not tipo_tabla: 
            continue
        
        ruta = os.path.join(PATH_LANDING, archivo)
        
        # Leemos solo la primera fila para contar columnas
        df_sample = pd.read_csv(
            ruta, 
            header=None, 
            sep=r'[;,]', 
            engine='python', 
            nrows=1, 
            encoding='latin-1'
        )
        
        esperadas = METADATA_FISICA[tipo_tabla]
        encontradas = len(df_sample.columns)
        
        # Ejecutamos la validación con los números correctos de origen
        pipeline_val.validate_data_quality(df_sample, tipo_tabla, esperadas)

# Ejecutar el control de calidad
try:
    task_data_quality_control()
    print("🚀 Estructura física validada. Los archivos coinciden con el origen esperado.")
except Exception as e:
    print(f"\n🛑 ARCHIVO CORRUPTO O INCOMPLETO: {e}")
    raise e

2026-03-29 11:20:16,132 - INFO - --- SISTEMA INICIALIZADO: Carpetas verificadas ---
🔍 Auditando 5 archivos físicos en landing...
2026-03-29 11:20:16,137 - INFO - ✅ Calidad OK para almacenes (2 columnas detectadas).
2026-03-29 11:20:16,139 - INFO - ✅ Calidad OK para creditos (6 columnas detectadas).
2026-03-29 11:20:16,142 - INFO - ✅ Calidad OK para departamentos (3 columnas detectadas).
2026-03-29 11:20:16,145 - INFO - ✅ Calidad OK para municipios (4 columnas detectadas).
2026-03-29 11:20:16,147 - INFO - ✅ Calidad OK para personas (14 columnas detectadas).
🚀 Estructura física validada. Los archivos coinciden con el origen esperado.


### Orquestación final - Ingesta de data en la BD

In [6]:
if __name__ == "__main__":
    pipeline = DataEngineerPipeline(DB_NAME)
    pipeline.run()

2026-03-29 11:20:16,154 - INFO - --- SISTEMA INICIALIZADO: Carpetas verificadas ---
2026-03-29 11:20:16,154 - INFO - Procesando archivo: almacenes.csv


C:\Users\boda1\AppData\Local\Temp\ipykernel_2044\1428928789.py:103: ParserWarning: Skipping line 17308: Expected 2 fields in line 17308, saw 3. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df = pd.read_csv(
C:\Users\boda1\AppData\Local\Temp\ipykernel_2044\1428928789.py:103: ParserWarning: Skipping line 18007: Expected 2 fields in line 18007, saw 3. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df = pd.read_csv(
C:\Users\boda1\AppData\Local\Temp\ipykernel_2044\1428928789.py:103: ParserWarning: Skipping line 24719: Expected 2 fields in line 24719, saw 3. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df = pd.read_csv(
C:\Users\boda1\AppData\Local\Temp\ipykernel_2044\1428928789.py:103: ParserWarning: Skipping line 27860: Expected 2 fields in line 27860, saw 3. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df

2026-03-29 11:20:18,661 - INFO - OK: almacenes.csv | 0 registros nuevos | 2.51 segundos.
2026-03-29 11:20:18,676 - INFO - Procesando archivo: creditos2.csv
2026-03-29 11:20:18,799 - INFO - OK: creditos2.csv | 0 registros nuevos | 0.12 segundos.
2026-03-29 11:20:18,799 - INFO - Procesando archivo: departamentos.csv
2026-03-29 11:20:18,883 - INFO - OK: departamentos.csv | 0 registros nuevos | 0.08 segundos.
2026-03-29 11:20:18,883 - INFO - Procesando archivo: municipios.csv
2026-03-29 11:20:18,982 - INFO - OK: municipios.csv | 0 registros nuevos | 0.10 segundos.
2026-03-29 11:20:18,982 - INFO - Procesando archivo: personas2.csv
2026-03-29 11:20:19,134 - INFO - OK: personas2.csv | 0 registros nuevos | 0.15 segundos.
